# Market Risk Dashboard
## VaR, Expected Shortfall, GARCH & Machine Learning

**Author:** Yanis Ait Ouamara  
**Program:** Master 2 Finance Technology Data — University Paris 1 Panthéon-Sorbonne  
**Period:** 2015-2024  

### Portfolio
| Ticker | Company | Sector | Geography |
|--------|---------|--------|-----------|
| AAPL | Apple | Technology | USA |
| JPM | JPMorgan | Banking | USA |
| MC.PA | LVMH | Luxury | France |
| NESN.SW | Nestlé | Consumer Staples | Switzerland |
| TTE.PA | TotalEnergies | Energy | France |

## Step 1 — Data Collection

In [ ]:
# Install required libraries
# !pip install yfinance arch xgboost shap

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Define tickers and period
tickers = ['AAPL', 'JPM', 'MC.PA', 'NESN.SW', 'TTE.PA']
start_date = '2015-01-01'
end_date = '2024-12-31'

# Download closing prices
prices = yf.download(tickers, start=start_date, end=end_date)['Close']

print(prices.shape)
print(prices.head())
print(prices.isnull().sum())

## Step 2 — Data Cleaning & Log Returns

In [ ]:
# Forward fill missing values
prices_clean = prices.ffill()

print('Missing values after cleaning:')
print(prices_clean.isnull().sum())

# Compute log-returns
returns = np.log(prices_clean / prices_clean.shift(1))
returns = returns.dropna()

print(f'\nDimensions: {returns.shape}')
print('\nDescriptive statistics:')
print(returns.describe())

## Step 3 — Visualize Returns (Volatility Clustering)

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(12, 15))

for i, ticker in enumerate(returns.columns):
    axes[i].plot(returns.index, returns[ticker],
                linewidth=0.5, color='steelblue')
    axes[i].set_title(f'{ticker} — Daily Log Returns')
    axes[i].set_ylabel('Return')
    axes[i].axhline(y=0, color='red',
                   linestyle='--', linewidth=0.5)

plt.tight_layout()
plt.savefig('returns_plot.png', dpi=150)
plt.show()
print('Volatility clustering is clearly visible during Covid (2020) and Ukraine war (2022)')

## Step 4 — Parametric VaR

In [ ]:
confidence_level = 0.95
alpha = 1 - confidence_level

var_parametric = {}

for ticker in returns.columns:
    mu = returns[ticker].mean()
    sigma = returns[ticker].std()
    z = norm.ppf(alpha)
    var = mu + z * sigma
    var_parametric[ticker] = var

print('Parametric VaR at 95%:')
for ticker, var in var_parametric.items():
    print(f'{ticker}: {var*100:.2f}%')

print('\nConclusion: Nestlé is the least risky asset (-1.65%)')
print('Limitation: assumes constant volatility and normal distribution')

## Step 5 — Historical VaR

In [ ]:
var_historical = {}

for ticker in returns.columns:
    var = returns[ticker].quantile(alpha)
    var_historical[ticker] = var

# Comparison
print('Comparison Parametric vs Historical VaR:')
print(f"{'Ticker':<10} {'Parametric':>12} {'Historical':>12}")
print('-' * 36)
for ticker in returns.columns:
    vp = var_parametric[ticker]
    vh = var_historical[ticker]
    print(f'{ticker:<10} {vp*100:>11.2f}% {vh*100:>11.2f}%')

print('\nConclusion: Parametric VaR slightly overestimates risk')
print('The 2015-2024 period was relatively benign despite Covid')

## Step 6 — Monte Carlo VaR

In [ ]:
np.random.seed(42)
n_simulations = 10000

var_montecarlo = {}

for ticker in returns.columns:
    mu = returns[ticker].mean()
    sigma = returns[ticker].std()
    simulated_returns = np.random.normal(mu, sigma, n_simulations)
    var = np.percentile(simulated_returns, 5)
    var_montecarlo[ticker] = var

# Comparison of 3 methods
print('Comparison of 3 VaR methods:')
print(f"{'Ticker':<10} {'Parametric':>12} {'Historical':>12} {'Monte Carlo':>13}")
print('-' * 50)
for ticker in returns.columns:
    vp = var_parametric[ticker]
    vh = var_historical[ticker]
    vm = var_montecarlo[ticker]
    print(f'{ticker:<10} {vp*100:>11.2f}% {vh*100:>11.2f}% {vm*100:>12.2f}%')

print('\nConclusion: Monte Carlo and Parametric are very close')
print('Both use the same assumptions (normal distribution, constant sigma)')

## Step 7 — Expected Shortfall (CVaR)

In [ ]:
es_historical = {}

for ticker in returns.columns:
    var = var_historical[ticker]
    tail_returns = returns[ticker][returns[ticker] <= var]
    es = tail_returns.mean()
    es_historical[ticker] = es

# Comparison VaR vs ES
print('Comparison Historical VaR vs Expected Shortfall:')
print(f"{'Ticker':<10} {'VaR 95%':>10} {'ES 95%':>10} {'Ratio ES/VaR':>14}")
print('-' * 48)
for ticker in returns.columns:
    var = var_historical[ticker]
    es = es_historical[ticker]
    ratio = es / var
    print(f'{ticker:<10} {var*100:>9.2f}% {es*100:>9.2f}% {ratio:>13.2f}x')

print('\nConclusion: ES averages 1.5x the VaR for all assets')
print('VaR alone underestimates tail risk — justifying Basel III shift to ES')

## Step 8 — GARCH(1,1) Dynamic Volatility

In [ ]:
from arch import arch_model

garch_volatility = {}
var_garch = {}

for ticker in returns.columns:
    r = returns[ticker] * 100
    model = arch_model(r, vol='Garch', p=1, q=1, dist='normal')
    result = model.fit(disp='off')
    cond_vol = result.conditional_volatility / 100
    garch_volatility[ticker] = cond_vol
    mu = returns[ticker].mean()
    z = norm.ppf(alpha)
    var_dynamic = mu + z * cond_vol
    var_garch[ticker] = var_dynamic
    print(f"{ticker} — omega: {result.params['omega']:.6f}, "
          f"alpha: {result.params['alpha[1]']:.4f}, "
          f"beta: {result.params['beta[1]']:.4f}, "
          f"alpha+beta: {result.params['alpha[1]']+result.params['beta[1]']:.4f}")

print('\nConclusion: All models stationary (alpha+beta < 1)')
print('TotalEnergies has highest persistence (0.99) — consistent with oil price exposure')

In [ ]:
# Visualize Dynamic VaR
fig, axes = plt.subplots(5, 1, figsize=(12, 15))

for i, ticker in enumerate(returns.columns):
    axes[i].plot(returns.index, returns[ticker]*100,
                color='grey', linewidth=0.5, label='Returns')
    axes[i].plot(var_garch[ticker].index, var_garch[ticker]*100,
                color='red', linewidth=1, label='VaR GARCH 95%')
    axes[i].set_title(f'{ticker} — Dynamic VaR GARCH')
    axes[i].set_ylabel('Return (%)')
    axes[i].legend(loc='lower right')

plt.tight_layout()
plt.savefig('var_garch_plot.png', dpi=150)
plt.show()

In [ ]:
# Comparison of all 4 VaR methods
print('Comparison of 4 VaR methods at 95%:')
print(f"{'Ticker':<10} {'Param':>8} {'Hist':>8} {'MC':>8} {'GARCH avg':>12}")
print('-' * 52)

for ticker in returns.columns:
    vp = var_parametric[ticker]
    vh = var_historical[ticker]
    vm = var_montecarlo[ticker]
    vg = var_garch[ticker].mean()
    print(f'{ticker:<10} {vp*100:>7.2f}% {vh*100:>7.2f}% {vm*100:>7.2f}% {vg*100:>11.2f}%')

print('\nConclusion: GARCH average is lower because calm periods dominate')
print('The key advantage of GARCH is its real-time adaptation — not its average')

## Step 9 — Machine Learning: Stress Day Prediction

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
import xgboost as xgb

# Equally weighted portfolio
portfolio_returns = returns.mean(axis=1)

# Portfolio VaR
var_portfolio = portfolio_returns.quantile(0.05)
print(f'Portfolio VaR: {var_portfolio*100:.2f}%')

# Target variable: 1 if stress day, 0 otherwise
y = (portfolio_returns < var_portfolio).astype(int)
print(f'\nClass distribution:')
print(y.value_counts())
print(f'% stress days: {y.mean()*100:.2f}%')

In [ ]:
# Build features
features = pd.DataFrame(index=returns.index)

# Lagged returns
features['portfolio_return_lag1'] = portfolio_returns.shift(1)
features['portfolio_return_lag2'] = portfolio_returns.shift(2)
features['portfolio_return_lag3'] = portfolio_returns.shift(3)

# Realized volatility
features['realized_vol_5d'] = portfolio_returns.rolling(5).std()
features['realized_vol_20d'] = portfolio_returns.rolling(20).std()

# GARCH volatility
portfolio_garch = pd.DataFrame(garch_volatility).mean(axis=1)
features['garch_vol'] = portfolio_garch.values

# Cumulative returns
features['return_5d'] = portfolio_returns.rolling(5).sum()
features['return_20d'] = portfolio_returns.rolling(20).sum()

# Align and clean
df_ml = features.copy()
df_ml['target'] = y
df_ml = df_ml.dropna()

X = df_ml.drop('target', axis=1)
y_clean = df_ml['target']

print(f'ML Dataset: {X.shape[0]} observations, {X.shape[1]} features')
print(f'Stress days: {y_clean.sum()}')
print(f'Features: {X.columns.tolist()}')

In [ ]:
# Train/Test split (no shuffle — time series)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_clean, test_size=0.2, shuffle=False)

print(f'Train: {X_train.shape[0]} observations')
print(f'Test: {X_test.shape[0]} observations')
print(f'Stress days in test: {y_test.sum()}')

# Random Forest
rf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

# XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    scale_pos_weight=len(y_train[y_train==0])/len(y_train[y_train==1]),
    random_state=42,
    eval_metric='logloss')
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

# Results
print('\nRandom Forest:')
print(classification_report(y_test, y_pred_rf))
print(f'AUC-ROC: {roc_auc_score(y_test, rf.predict_proba(X_test)[:,1]):.4f}')

print('\nXGBoost:')
print(classification_report(y_test, y_pred_xgb))
print(f'AUC-ROC: {roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:,1]):.4f}')

In [ ]:
# Visualize stress probabilities
prob_stress_xgb = xgb_model.predict_proba(X_test)[:,1]
prob_stress_rf = rf.predict_proba(X_test)[:,1]

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

axes[0].plot(X_test.index, prob_stress_xgb,
            color='red', linewidth=0.8, label='Stress probability')
axes[0].axhline(y=0.5, color='black', linestyle='--', linewidth=0.5)
axes[0].scatter(X_test.index[y_test==1], prob_stress_xgb[y_test==1],
               color='red', s=50, zorder=5, label='Actual stress days')
axes[0].set_title('XGBoost — Stress Day Probability')
axes[0].set_ylabel('Probability')
axes[0].legend()

axes[1].plot(X_test.index, prob_stress_rf,
            color='blue', linewidth=0.8, label='Stress probability')
axes[1].axhline(y=0.5, color='black', linestyle='--', linewidth=0.5)
axes[1].scatter(X_test.index[y_test==1], prob_stress_rf[y_test==1],
               color='red', s=50, zorder=5, label='Actual stress days')
axes[1].set_title('Random Forest — Stress Day Probability')
axes[1].set_ylabel('Probability')
axes[1].legend()

plt.tight_layout()
plt.savefig('stress_probability.png', dpi=150)
plt.show()
print('Conclusion: XGBoost clearly outperforms Random Forest')
print('Actual stress days correspond to probability peaks in XGBoost')

## Step 10 — SHAP Explainability

In [ ]:
import shap

# SHAP values for XGBoost
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

# Feature Importance
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test,
                 plot_type='bar', show=False)
plt.title('SHAP Feature Importance — XGBoost')
plt.tight_layout()
plt.savefig('shap_importance.png', dpi=150)
plt.show()

# Beeswarm Plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_test, show=False)
plt.title('SHAP Beeswarm Plot — XGBoost')
plt.tight_layout()
plt.savefig('shap_beeswarm.png', dpi=150)
plt.show()

print('Top 3 predictors of stress days:')
print('1. return_5d — 5-day cumulative return (most important)')
print('2. realized_vol_5d — short-term volatility')
print('3. realized_vol_20d — medium-term volatility')
print('\nConclusion: Recent negative returns + high volatility = strongest stress signal')

## Step 11 — Backtesting: Kupiec & Christoffersen

In [ ]:
def kupiec_test(returns, var, confidence=0.95):
    alpha = 1 - confidence
    exceptions = (returns < var).astype(int)
    n = len(exceptions)
    x = exceptions.sum()
    p_hat = x / n
    if p_hat == 0 or p_hat == 1:
        return None, None, x, p_hat
    lr = -2 * np.log(
        ((1-alpha)**(n-x) * alpha**x) /
        ((1-p_hat)**(n-x) * p_hat**x))
    p_value = 1 - stats.chi2.cdf(lr, df=1)
    return lr, p_value, x, p_hat


def christoffersen_test(returns, var):
    exceptions = (returns < var).astype(int).values
    n00, n01, n10, n11 = 0, 0, 0, 0
    for i in range(1, len(exceptions)):
        if exceptions[i-1] == 0 and exceptions[i] == 0:
            n00 += 1
        elif exceptions[i-1] == 0 and exceptions[i] == 1:
            n01 += 1
        elif exceptions[i-1] == 1 and exceptions[i] == 0:
            n10 += 1
        elif exceptions[i-1] == 1 and exceptions[i] == 1:
            n11 += 1
    if (n00 + n01) == 0 or (n10 + n11) == 0:
        return None, None
    p01 = n01 / (n00 + n01)
    p11 = n11 / (n10 + n11)
    p = (n01 + n11) / (n00 + n01 + n10 + n11)
    if p == 0 or p == 1 or p01 == 0 or p11 == 0:
        return None, None
    lr = -2 * np.log(
        ((1-p)**(n00+n10) * p**(n01+n11)) /
        ((1-p01)**n00 * p01**n01 *
         (1-p11)**n10 * p11**n11))
    p_value = 1 - stats.chi2.cdf(lr, df=1)
    return lr, p_value


# Apply tests
print('=' * 65)
print('BACKTESTING RESULTS — Historical VaR at 95%')
print('=' * 65)
print(f"{'Ticker':<10} {'Exceptions':>12} {'Rate':>8} "
      f"{'Kupiec p':>12} {'Christoff p':>13} {'Verdict'}")
print('-' * 65)

for ticker in returns.columns:
    var = var_historical[ticker]
    r = returns[ticker]
    lr_k, pval_k, n_exc, rate = kupiec_test(r, var)
    lr_c, pval_c = christoffersen_test(r, var)
    if pval_k and pval_k > 0.05:
        if pval_c and pval_c > 0.05:
            verdict = 'PASS'
        else:
            verdict = 'CLUSTERING'
    else:
        verdict = 'FAIL'
    print(f'{ticker:<10} {n_exc:>12} {rate*100:>7.2f}% '
          f'{pval_k:>12.4f} {pval_c:>13.4f} {verdict}')

print('\nH0 Kupiec: exception rate = 5%')
print('H0 Christoffersen: exceptions are independent')
print('Reject H0 if p-value < 0.05')
print('\nConclusion: VaR correctly calibrated in frequency (Kupiec)')
print('But exceptions cluster during crises for 4/5 assets (Christoffersen)')
print('Only Nestlé passes both tests — confirming its defensive profile')

## Global Conclusion

This project demonstrates three fundamental insights:

1. **Static VaR methods are insufficient** — they are correctly calibrated on average (Kupiec ✅) but fail to capture volatility clustering (Christoffersen ⚠️).

2. **GARCH improves risk measurement** — dynamic VaR adapts in real time to market regimes, expanding during crises and contracting during calm periods.

3. **Machine Learning adds value** — XGBoost predicts stress days with AUC-ROC of 0.986. SHAP identifies 5-day returns and short-term volatility as the strongest predictors. Combined with GARCH, ML provides an early warning system complementary to classical VaR measures.

**Key message:** You cannot rely on a single risk measure. VaR gives the threshold, ES gives the severity, GARCH gives the dynamics, and Machine Learning gives the early warning signal. Together they form a complete market risk framework.